In [2]:
import os
import sys
import random
import copy
import re
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import AutoModel, AutoTokenizer
from sklearn.metrics import classification_report, accuracy_score, f1_score
from sklearn.utils.class_weight import compute_class_weight
from tqdm.auto import tqdm

# Add project root to path for src imports
repo_root = os.path.abspath(os.path.join(os.path.dirname('__file__'), '../..'))
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

from src.utils import seed_everything, load_config
seed_everything(1234)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device configured: {device} | Seed: {seed_val}')


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.5/8.5 MB 70.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 615.4/615.4 kB 28.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 235.8/235.8 kB 18.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 41.8 MB/s eta 0:00:00
✅ Using device: cuda


In [3]:
# Initialize OntologyEngine using relative path to ontology RDF
from src.ontology_engine import OntologyEngine, VSFCOntologyEngine
from src.utils import load_config

config_path = os.path.join(repo_root, 'configs/vsmec.yaml')
config = load_config(config_path)
onto_cfg = config.get('ontology', {})

ontology_path = os.path.join(repo_root, onto_cfg.get('rdf_path', 'ontology/ekman_appraisal_ontology.rdf'))

ontengine = OntologyEngine(
    rdf_path=ontology_path,
    default_conf=onto_cfg.get('default_conf', 0.85),
    manual_boost=onto_cfg.get('manual_boost', 1.2),
    negation_attenuation=onto_cfg.get('negation_attenuation', 0.5),
    alpha_similarity=onto_cfg.get('alpha_similarity', 0.2),
    verbose=True,
)


🚀 Initializing OntologyEngine v12.2 for VSMEC (Deep Inference) with /kaggle/input/kg-ekman6/ekman6_4_8.rdf...
📥 Loading KG for VSMEC: /kaggle/input/kg-ekman6/ekman6_4_8.rdf
   ✅ Loaded 90420 triples.
   📚 Lexicon size: 2540 entries (v12.2 VSMEC-Ready)


In [4]:
# Thử test câu này để ép code chạy Soft Negation
t_test = "không thực sự thích" 
v, _, tr = ontengine.getvector(t_test, debug=True)

print(f"Matches: {[x['phrase'] for x in tr]}") 
# Nếu kết quả ra: ['không', 'thích'] -> Soft Negation đã chạy!
print(f"Negation Flag: {v[-2]}") # Sẽ ra 1.0

Matches: ['không', 'thích']
Negation Flag: 1.0


In [5]:
# =========================================================
# CELL TÙY CHỌN: TRỰC QUAN HÓA VECTOR ONTOLOGY 24 CHIỀU
# Mục tiêu: Xem chi tiết 24 giá trị trong vector biểu diễn gì
# =========================================================
import numpy as np
import pandas as pd

# 1. Nhập câu bạn muốn kiểm tra
sample_text = "không phải là không vui"
print(f"📝 Câu input: '{sample_text}'\n")

# 2. Gọi hàm getvector (để debug=True để lấy thông tin giải nghĩa)
vec, flat_tokens, trace = ontengine.getvector(sample_text, debug=True)

print("🔍 CÁC TỪ KHÓA ĐƯỢC MATCH BỞI ONTOLOGY:")
if not trace:
    print(" - Không có từ khóa nào khớp với KG.")
for t in trace:
    print(f" - '{t['phrase']}' -> Score: {t['data']['score']:.2f}")
print("-" * 60)

# 3. Tái tạo lại cấu trúc 24 chiều để ánh xạ tên
# Theo code chuẩn của bạn: 7 Emotion + 9 Appraisal + 3 Intensity + 3 Polarity + 2 Negation
features = []
features.extend([f"Emotion: {e}" for e in ontengine.ALLEMOTIONS])
features.extend([f"Appraisal: {a.replace('Appraisal', '')}" for a in ontengine.ALLAPPRAISALS])
features.extend([f"Intensity: {i.replace('Intensity', '')}" for i in ontengine.INTENSITY_CLASSES])
features.extend([f"Polarity: {p.replace('Polarity', '')}" for p in ontengine.POLARITY_CLASSES])
features.extend(["Negation: Has_Negation", "Negation: Padding"])

# 4. In ra dạng mảng List raw như yêu cầu
print("📊 VECTOR ONTOLOGY RAW (24-D):")
print(f"vectorOntology: {np.round(vec, 4).tolist()}\n")

# 5. In ra dạng bảng chi tiết (chỉ lấy các feature được kích hoạt > 0)
df_vector = pd.DataFrame({
    "Index": range(0, 24),
    "Feature_Category": features,
    "Value": np.round(vec, 4)
})

active_features = df_vector[df_vector["Value"] > 0].copy()
print("✨ CÁC ĐẶC TRƯNG ĐƯỢC KÍCH HOẠT TRONG MẠNG LƯỚI (Value > 0):")
if active_features.empty:
    print(" (Vector toàn số 0 - Neutral tuyệt đối)")
else:
    print(active_features.to_string(index=False))
print("=" * 60)

📝 Câu input: 'không phải là không vui'

🔍 CÁC TỪ KHÓA ĐƯỢC MATCH BỞI ONTOLOGY:
 - 'không' -> Score: 0.95
 - 'phải' -> Score: 0.29
 - 'không' -> Score: 0.95
 - 'vui' -> Score: 0.90
------------------------------------------------------------
📊 VECTOR ONTOLOGY RAW (24-D):
vectorOntology: [0.17110000550746918, 0.0, 0.0, 0.49410000443458557, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.451200008392334, 0.14399999380111694, 0.0, 1.0, 0.0]

✨ CÁC ĐẶC TRƯNG ĐƯỢC KÍCH HOẠT TRONG MẠNG LƯỚI (Value > 0):
 Index       Feature_Category  Value
     0         Emotion: Anger 0.1711
     3     Emotion: Happiness 0.4941
    19     Polarity: Positive 0.4512
    20     Polarity: Negative 0.1440
    22 Negation: Has_Negation 1.0000


In [6]:
# =========================================================
# CELL: DIAGNOSTIC TEST SUITE CHO KG ENGINE V11.0
# Mục tiêu: Kiểm tra tự động 4 tính năng: 
# 1. Soft Negation (Giảm điểm) 
# 2. Polarity Flip (Đảo cực tính) 
# 3. Blacklist (Lọc nhiễu) 
# 4. Scope Fix (Xuyên từ rác)
# =========================================================
import pandas as pd
import numpy as np

# --- 1. HÀM GIẢI MÃ VECTOR (Vector -> Text) ---
def decode_vector(vec, engine):
    # 7 chiều đầu: Emotions
    emo_vals = vec[:7]
    if np.max(emo_vals) > 0.01:
        top_idx = np.argmax(emo_vals)
        emotion = engine.ALLEMOTIONS[top_idx]
        score = emo_vals[top_idx]
    else:
        emotion = "Neutral/None"
        score = 0.0
        
    # Index 19, 20, 21: Polarity [Pos, Neg, Neu]
    pol_vals = vec[19:22]
    pol_labels = ["Positive", "Negative", "Neutral"]
    if np.max(pol_vals) > 0:
        polarity = pol_labels[np.argmax(pol_vals)]
    else:
        polarity = "Unknown" # Nếu vẫn ra Unknown tức là KG chưa gán Polarity cho từ đó
        
    has_neg = vec[23] > 0
    return emotion, score, polarity, has_neg
    
# --- 2. ĐỊNH NGHĨA TEST CASES ---
# Format: (Input Text, Kỳ vọng Emotion, Kỳ vọng Polarity, Mô tả lỗi cần check)
test_cases = [
    # --- NHÓM 1: CƠ BẢN (Sanity Check) ---
    ("món này ngon tuyệt", "Happiness", "Positive", "Basic Positive"),
    ("thất vọng tràn trề", "Sadness", "Negative", "Basic Negative"),

    # --- NHÓM 2: SOFT NEGATION (Quan trọng nhất) ---
    # v11.0: Emotion phải giảm điểm (về gần 0 hoặc Neutral), Polarity phải là Negative
    ("tôi không thích nó", "Neutral/None", "Negative", "Soft Negation (Không thích)"),
    ("chẳng vui chút nào", "Neutral/None", "Negative", "Soft Negation (Chẳng vui)"),

    # --- NHÓM 3: SCOPE FIX (Lỗi 'hề') ---
    # Phải bắt được phủ định dù có từ đệm
    ("tôi không hề thích", "Neutral/None", "Negative", "Scope Skip (Không hề...)"),
    ("không phải là ghét", "Neutral/None", "Positive", "Double Negation (Không ghét -> Pos)"),

    # --- NHÓM 4: BLACKLIST (Lỗi từ đa nghĩa) ---
    # Không được bắt chữ "hay" (hoặc) và "chả" (món ăn)
    ("làm hay nghỉ", "Neutral/None", "Unknown", "Blacklist: Hay (Or)"),
    ("bún chả rất ngon", "Happiness", "Positive", "Blacklist: Chả (Food)"),
    ("tôi là sinh viên", "Neutral/None", "Unknown", "Blacklist: Tôi/Là (Pronouns)")
]

# --- 3. CHẠY TEST TỰ ĐỘNG ---
print(f"{'='*25} BÁO CÁO KIỂM THỬ KG V11.0 {'='*25}")
results = []
pass_count = 0

for text, exp_emo, exp_pol, desc in test_cases:
    # Chạy Engine
    vec, _, trace = ontengine.getvector(text, debug=True)
    out_emo, out_score, out_pol, out_neg = decode_vector(vec, ontengine)
    
    # LOGIC CHẤM ĐIỂM (PASS/FAIL)
    status = "❌ FAIL"
    note = ""
    
    # Check 1: Emotion
    # Nếu kỳ vọng là Neutral/None -> Score thực tế phải thấp (< 0.5) hoặc label là Neutral
    emo_pass = False
    if exp_emo == "Neutral/None":
        if out_score < 0.5 or out_emo == "Neutral": emo_pass = True
    else:
        if out_emo == exp_emo and out_score > 0.3: emo_pass = True # Score > 0.3 là active
        
    # Check 2: Polarity (Rất quan trọng cho Sentiment 3 nhãn)
    pol_pass = False
    if exp_pol == "Unknown":
        pol_pass = True # Không care
    elif out_pol == exp_pol:
        pol_pass = True
        
    # Đặc biệt: Case "bún chả" -> Fail nếu bắt nhầm negation
    if "Blacklist: Chả" in desc and out_neg:
        status = "❌ FAIL (False Negation)"
    # Đặc biệt: Case "không thích" -> Fail nếu vẫn là Positive
    elif "Soft Negation" in desc and out_pol == "Positive":
        status = "❌ FAIL (Wrong Polarity)"
    elif emo_pass and pol_pass:
        status = "✅ PASS"
        pass_count += 1
    
    # Log kết quả
    matched_str = ", ".join([t['phrase'] for t in trace])
    results.append({
        "Scenario": desc,
        "Input": text,
        "Matched": matched_str,
        "Prediction": f"{out_emo} ({out_score:.2f}) | {out_pol}",
        "NegFlag": "YES" if out_neg else "NO",
        "Status": status
    })

# --- 4. HIỂN THỊ KẾT QUẢ ---
df_res = pd.DataFrame(results)
# Chỉ hiển thị các cột quan trọng
print(df_res[["Input", "Matched", "Prediction", "Status"]].to_string(index=False))

print("-" * 70)
print(f"🏆 KẾT QUẢ: {pass_count}/{len(test_cases)} Test Cases Passed.")

if pass_count == len(test_cases):
    print("🚀 KẾT LUẬN: KG Engine v11.0 hoạt động HOÀN HẢO. Sẵn sàng train!")
else:
    print("⚠️ KẾT LUẬN: Vẫn còn lỗi (xem các dòng FAIL).")

========================= BÁO CÁO KIỂM THỬ KG V11.0 =========================
             Input           Matched                    Prediction                  Status
món này ngon tuyệt             tuyệt   Happiness (0.75) | Positive                  ✅ PASS
thất vọng tràn trề         thất vọng     Sadness (0.58) | Negative                  ✅ PASS
tôi không thích nó       không thích     Disgust (0.55) | Negative                  ❌ FAIL
chẳng vui chút nào               vui   Happiness (0.79) | Positive ❌ FAIL (Wrong Polarity)
tôi không hề thích      không, thích   Happiness (0.45) | Positive                  ❌ FAIL
không phải là ghét không, phải, ghét       Anger (0.17) | Negative                  ❌ FAIL
      làm hay nghỉ              nghỉ     Sadness (0.39) | Negative                  ✅ PASS
  bún chả rất ngon               rất   Happiness (0.42) | Positive                  ✅ PASS
  tôi là sinh viên                   Neutral/None (0.00) | Unknown                  ✅ PASS
------------

In [7]:
# ---------------------------------------------------------
# CELL 3: DATASET & PREPROCESSING (VSMEC)
# ---------------------------------------------------------
tokenizer = AutoTokenizer.from_pretrained("vinai/phobert-base-v2")

class EmotionDataset(Dataset):
    def __init__(self, df, tokenizer, engine, max_len=128):
        self.df = df
        self.tokenizer = tokenizer
        self.engine = engine
        self.max_len = max_len
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        txt = str(self.df.iloc[idx]['Sentence'])
        lbl = self.df.iloc[idx]['label_idx']
        enc = self.tokenizer.encode_plus(txt, max_length=self.max_len, padding='max_length', truncation=True, return_tensors='pt')
        vec = self.engine.getvector(txt)
        return {'input_ids': enc['input_ids'].flatten(), 'attention_mask': enc['attention_mask'].flatten(), 'ontology_features': torch.tensor(vec, dtype=torch.float), 'labels': torch.tensor(lbl, dtype=torch.long)}

try:
    # VSMEC Path
    base = "../../data/vsmec"
    df_train = pd.read_excel(f"{base}/train_nor_811.xlsx")
    df_valid = pd.read_excel(f"{base}/valid_nor_811.xlsx")
    df_test = pd.read_excel(f"{base}/test_nor_811.xlsx")
    
    # Đổi tên cột cho đồng nhất
    for df in [df_train, df_valid, df_test]:
        df.dropna(subset=['Sentence', 'Emotion'], inplace=True)
        # Map 'Other' thành 'Other' (hoặc bạn có thể map thành Neutral nếu muốn)
    print("✅ VSMEC Data Loaded")
except Exception as e: 
    print(f"❌ Load Error: {e}")

# Label Encoding
labels = sorted(df_train['Emotion'].unique().astype(str))
le = LabelEncoder()
le.fit(labels)
print(f"📋 Classes: {le.classes_}")

for df in [df_train, df_valid, df_test]:
    df['label_idx'] = le.transform(df['Emotion'].astype(str))

train_loader = DataLoader(EmotionDataset(df_train, tokenizer, ontengine), batch_size=32, shuffle=True)
valid_loader = DataLoader(EmotionDataset(df_valid, tokenizer, ontengine), batch_size=32)
test_loader = DataLoader(EmotionDataset(df_test, tokenizer, ontengine), batch_size=32)

# Class Weights
cls_w = compute_class_weight('balanced', classes=np.unique(df_train['label_idx']), y=df_train['label_idx'])
weights_tensor = torch.tensor(cls_w, dtype=torch.float).to(device)
print(f"⚖️ Weights: {weights_tensor}")

config.json:   0%|          | 0.00/678 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

bpe.codes: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

✅ VSMEC Data Loaded
📋 Classes: ['Anger' 'Disgust' 'Enjoyment' 'Fear' 'Other' 'Sadness' 'Surprise']
⚖️ Weights: tensor([2.0270, 0.7400, 0.5087, 2.4924, 0.7763, 0.8369, 3.2751],
       device='cuda:0')


In [8]:
# Import PhoBERT Fusion architecture from src.models
from src.models.phobert_fusion_models import PhoBERT_Fusion_V2

try:
    dummy_vec = ontengine.getvector('test')
    if isinstance(dummy_vec, tuple): dummy_vec = dummy_vec[0]
    ONT_INPUT_DIM = len(dummy_vec)
except:
    ONT_INPUT_DIM = 24
print(f'✅ Loaded PhoBERT_Fusion_V2 | Detected Ontology Dimension: {ONT_INPUT_DIM}')


In [9]:
# ---------------------------------------------------------
# CELL 5: TRAINER
# ---------------------------------------------------------
def train_experiment(model, name, epochs=15, patience=4):
    print(f"\n🚀 START: {name}")
    if torch.cuda.device_count() > 1: model = nn.DataParallel(model)
    model.to(device)
    optim = torch.optim.AdamW(model.parameters(), lr=2e-5)
    crit = nn.CrossEntropyLoss(weight=weights_tensor)
    
    best_f1, pat = 0, 0
    best_w = copy.deepcopy(model.state_dict()) if not isinstance(model, nn.DataParallel) else copy.deepcopy(model.module.state_dict())
    
    for ep in range(epochs):
        model.train()
        loss_acc = 0
        for b in train_loader:
            optim.zero_grad()
            ids, m, o, l = [b[k].to(device) for k in ['input_ids','attention_mask','ontology_features','labels']]
            loss = crit(model(ids, m, o), l)
            loss.backward(); optim.step()
            loss_acc += loss.item()
            
        acc, f1 = evaluate(model, valid_loader)
        print(f"Ep {ep+1} | Loss: {loss_acc/len(train_loader):.4f} | Val F1: {f1:.4f}", end=" ")
        
        if f1 > best_f1:
            best_f1 = f1; pat = 0; best_w = copy.deepcopy(model.state_dict()) if not isinstance(model, nn.DataParallel) else copy.deepcopy(model.module.state_dict())
            print("✅")
        else:
            pat += 1
            print(f"⚠️ {pat}")
        if pat >= patience: break
    
    if isinstance(model, nn.DataParallel): model.module.load_state_dict(best_w)
    else: model.load_state_dict(best_w)
    return model

def evaluate(model, loader):
    model.eval()
    p, t = [], []
    with torch.no_grad():
        for b in loader:
            ids, m, o, l = [b[k].to(device) for k in ['input_ids','attention_mask','ontology_features','labels']]
            _, pr = torch.max(model(ids, m, o), 1)
            p.extend(pr.cpu().numpy()); t.extend(l.cpu().numpy())
    return accuracy_score(t, p), f1_score(t, p, average='macro')

In [10]:
# ---------------------------------------------------------
# CELL 6: MAIN EXPERIMENTS (5 CASES - VSMEC)
# ---------------------------------------------------------
NC = len(le.classes_)

# 1. Baseline
m1 = PhoBERT_Fusion_V2(NC, 'none')
m1 = train_experiment(m1, "Baseline (PhoBERT)", epochs=15)

# 3. Raw + Gate
m3 = PhoBERT_Fusion_V2(NC, 'gate', ontology_dim=None)
m3 = train_experiment(m3, "Raw + Gate", epochs=15)

2026-04-16 14:53:41.427879: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1776351221.617380      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1776351221.671501      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1776351222.119774      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776351222.119812      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776351222.119814      55 computation_placer.cc:177] computation placer alr

pytorch_model.bin:   0%|          | 0.00/540M [00:00<?, ?B/s]

Some weights of RobertaModel were not initialized from the model checkpoint at vinai/phobert-base-v2 and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.



🚀 START: Baseline (PhoBERT)


model.safetensors:   0%|          | 0.00/540M [00:00<?, ?B/s]

KeyboardInterrupt: 

In [ ]:
import torch
import os
from IPython.display import FileLink

# Tạo thư mục lưu trữ
output_dir = "/kaggle/working/vsmec_checkpoints"
os.makedirs(output_dir, exist_ok=True)

# 1. Lưu checkpoint cho m1 (Baseline VSMEC)
if 'm1' in globals():
    m1_path = os.path.join(output_dir, "vsmec_baseline_m1.pth")
    torch.save(m1.state_dict(), m1_path)
    print(f"✅ Đã lưu m1 tại: {m1_path}")
    display(FileLink(r'vsmec_checkpoints/vsmec_baseline_m1.pth'))
else:
    print("⚠️ Biến 'm1' chưa được khởi tạo. Hãy chạy Cell huấn luyện m1 trước.")

# 2. Lưu checkpoint cho m3 (Raw + Gate VSMEC)
if 'm3' in globals():
    m3_path = os.path.join(output_dir, "vsmec_gate_m3.pth")
    torch.save(m3.state_dict(), m3_path)
    print(f"✅ Đã lưu m3 tại: {m3_path}")
    display(FileLink(r'vsmec_checkpoints/vsmec_gate_m3.pth'))
else:
    print("⚠️ Biến 'm3' chưa được khởi tạo. Hãy chạy Cell huấn luyện m3 trước.")

In [ ]:
# ---------------------------------------------------------
# CELL 6: MAIN EXPERIMENTS (5 CASES - VSMEC)
# ---------------------------------------------------------
NC = len(le.classes_)

# 1. Baseline
m1 = PhoBERT_Fusion_V2(NC, 'none')
m1 = train_experiment(m1, "Baseline (PhoBERT)", epochs=15)



In [ ]:
# =========================================================
# CELL: EVALUATION - BASELINE vs. NEURO-SYMBOLIC (VSMEC 7-LABELS)
# Logic: Sử dụng m1 + Rule-based Post-processing từ Ontology
# =========================================================
import torch
import torch.nn.functional as F
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score, classification_report
from tqdm.auto import tqdm

class NeuroSymbolicReasonerVSMEC:
    def __init__(self, model, engine, tokenizer, device, 
                 ontology_threshold=0.5, model_conf_limit=0.7):
        self.model = model
        self.engine = engine
        self.tokenizer = tokenizer
        self.device = device
        self.ont_threshold = ontology_threshold
        self.model_conf_limit = model_conf_limit
        # Map index của model về nhãn cảm xúc thực tế
        self.idx2label = {i: label for i, label in enumerate(le.classes_)}

    def infer(self, text):
        # B1: Neural Inference (Dùng model m1)
        inputs = self.tokenizer(text, return_tensors="pt", truncation=True, max_length=128, padding='max_length')
        input_ids = inputs['input_ids'].to(self.device)
        attention_mask = inputs['attention_mask'].to(self.device)
        
        # Lấy vector KG từ Ontology Engine
        # Fix lỗi unpack: getvector trả về (fin, flat, hit_trace) khi debug=True
        vec, _, _ = self.engine.getvector(text, debug=True)
        ont_tensor = torch.tensor(np.array([vec]), dtype=torch.float).to(self.device)
        
        self.model.eval()
        with torch.no_grad():
            # M1 là PhoBERT không dùng feature ontology nên ta chỉ truyền ids và mask
            # Lưu ý: Theo cấu trúc PhoBERT_Fusion_V2, nếufusion_type='none' thì tham số thứ 3 (ont) không dùng
            outputs = self.model(input_ids, attention_mask, ont_tensor)
            probs = F.softmax(outputs, dim=1)
            
        pred_idx = torch.argmax(probs).item()
        confidence = probs[0][pred_idx].item()
        neural_label = self.idx2label[pred_idx]

        # B2: Symbolic Override (Hậu xử lý bằng luật)
        # Lấy cảm xúc mạnh nhất từ Ontology (7 chiều đầu của vector)
        ont_emotions = vec[0:7]
        max_ont_idx = np.argmax(ont_emotions)
        max_ont_score = ont_emotions[max_ont_idx]
        ont_label_name = self.engine.ALLEMOTIONS[max_ont_idx]
        
        # Mapping nhãn từ Ontology sang nhãn VSMEC (Happiness -> Enjoyment, Neutral -> Other)
        ont_to_vsmec = {
            "Happiness": "Enjoyment",
            "Neutral": "Other"
        }
        final_ont_label = ont_to_vsmec.get(ont_label_name, ont_label_name)
        
        final_label = neural_label
        
        # RULE: Nếu Model dự đoán là 'Other' nhưng Ontology bắt được cảm xúc cụ thể rất mạnh
        # Hoặc Model thiếu tự tin (confidence thấp) và Ontology có tín hiệu rõ ràng
        if neural_label == "Other" or confidence < self.model_conf_limit:
            if max_ont_score > self.ont_threshold:
                # Tìm index của nhãn Ontology trong danh sách nhãn VSMEC
                try:
                    target_idx = list(le.classes_).index(final_ont_label)
                    final_label = final_ont_label
                except ValueError:
                    pass # Nhãn không tồn tại trong VSMEC thì bỏ qua

        return neural_label, final_label

# 2. Chạy Đánh giá trên tập Test của VSMEC
print(f"🚀 Bắt đầu đánh giá Neuro-Symbolic trên {len(df_test)} mẫu test VSMEC...")

# Khởi tạo Reasoner với model Baseline (m1)
reasoner_vsmec = NeuroSymbolicReasonerVSMEC(
    model=m1, 
    engine=ontengine, 
    tokenizer=tokenizer, 
    device=device,
    ontology_threshold=0.4, # Ngưỡng nhạy hơn cho 7 nhãn
    model_conf_limit=0.65
)

y_true = []
y_pred_base = []
y_pred_neuro = []

for _, row in tqdm(df_test.iterrows(), total=len(df_test)):
    text = row['Sentence']
    true_label_str = row['Emotion']
    
    base_lbl, neuro_lbl = reasoner_vsmec.infer(text)
    
    y_true.append(true_label_str)
    y_pred_base.append(base_lbl)
    y_pred_neuro.append(neuro_lbl)

# 3. Tính Metrics & Báo cáo
acc_base = accuracy_score(y_true, y_pred_base)
f1_base = f1_score(y_true, y_pred_base, average='macro')

acc_neuro = accuracy_score(y_true, y_pred_neuro)
f1_neuro = f1_score(y_true, y_pred_neuro, average='macro')

results_df = pd.DataFrame({
    "Metric": ["Accuracy", "F1-Macro"],
    "Baseline (m1)": [acc_base, f1_base],
    "M1 + Ontology Rules": [acc_neuro, f1_neuro],
    "Improvement": [acc_neuro - acc_base, f1_neuro - f1_base]
})

print("\n" + "="*60)
print("🏆 KẾT QUẢ VSMEC: BASELINE vs RULE-BASED POST-PROCESSING")
print("="*60)
print(results_df.round(4).to_markdown(index=False))

print("\n🧠 Chi tiết M1 + Ontology Rules:")
print(classification_report(y_true, y_pred_neuro, target_names=le.classes_, digits=4))

# Đếm số ca đã sửa
changes = sum(1 for b, n in zip(y_pred_base, y_pred_neuro) if b != n)
print(f"\n🔄 Tổng số mẫu đã được Ontology can thiệp sửa lại: {changes}/{len(df_test)} ({changes/len(df_test)*100:.2f}%)")

In [ ]:
# 2. Raw + Concat (xAI)
m2 = PhoBERT_Fusion_V2(NC, 'concat', ontology_dim=None)
m2 = train_experiment(m2, "Raw + Concat", epochs=15)

# 3. Raw + Gate
m3 = PhoBERT_Fusion_V2(NC, 'gate', ontology_dim=None)
m3 = train_experiment(m3, "Raw + Gate", epochs=15)



In [ ]:
# 4. Dense (64d) + Concat
m4 = PhoBERT_Fusion_V2(NC, 'concat', ontology_dim=64)
m4 = train_experiment(m4, "Dense(64) + Concat", epochs=15)

# 5. Dense (64d) + Gate
m5 = PhoBERT_Fusion_V2(NC, 'gate', ontology_dim=64)
m5 = train_experiment(m5, "Dense(64) + Gate", epochs=15)

In [ ]:
# ---------------------------------------------------------
# CELL 7: FINAL REPORT
# ---------------------------------------------------------
def gp(model, loader):
    model.eval()
    p, t = [], []
    with torch.no_grad():
        for b in loader:
            ids, m, o, l = [b[k].to(device) for k in ['input_ids','attention_mask','ontology_features','labels']]
            _, pr = torch.max(model(ids, m, o), 1)
            p.extend(pr.cpu().numpy()); t.extend(l.cpu().numpy())
    return t, p

y_true, p1 = gp(m1, test_loader)
_, p2 = gp(m2, test_loader)
_, p3 = gp(m3, test_loader)
_, p4 = gp(m4, test_loader)
_, p5 = gp(m5, test_loader)

res = {
    "Model": ["Baseline", "Raw+Concat", "Raw+Gate", "Dense+Concat", "Dense+Gate"],
    "F1-Macro": [f1_score(y_true, p, average='macro') for p in [p1, p2, p3, p4, p5]],
    "Accuracy": [accuracy_score(y_true, p) for p in [p1, p2, p3, p4, p5]]
}

print("\n" + "="*50)
print("🏆 VSMEC FINAL RESULTS")
print("="*50)
print(pd.DataFrame(res).round(4).to_markdown(index=False))

print("\n" + "="*50)
print("📑 DETAILED REPORT (DENSE + GATE)")
print("="*50)
print(classification_report(y_true, p5, target_names=le.classes_, digits=4))

In [ ]:
import torch
import torch.nn as nn
from transformers import AutoModel
import sys

# ---------------------------------------------------------
# 1. ĐỊNH NGHĨA MODEL: DEEP ONTOLOGY FUSION
# ---------------------------------------------------------

# Tự động lấy số lượng class (nếu có biến le từ các cell trước)
try:
    NUM_CLASSES = len(le.classes_)
except NameError:
    NUM_CLASSES = 7  # Mặc định cho VSFC

# Tự động lấy kích thước vector ontology (nếu có từ cell trước)
try:
    ONT_DIM = ONT_INPUT_DIM 
except NameError:
    ONT_DIM = 24 # Mặc định

from src.models.phobert_fusion_models import PhoBERT_DeepOntology

# ---------------------------------------------------------
# 2. THỰC NGHIỆM BỔ SUNG: WIDE DEEP ONTOLOGY (24 -> 768 -> 768)
# ---------------------------------------------------------

# === CASE C: WIDE DEEP (768d) + CONCAT ===
print("\n" + "="*40)
print("🧪 CASE C: Wide Deep (24->768->768) + CONCAT")
print("="*40)

# Khởi tạo model với hidden_dim = 768 (Wide) thay vì 64 (Bottleneck)
model_wide_concat = PhoBERT_DeepOntology(
    n_classes=NUM_CLASSES, 
    fusion_type='concat', 
    ont_input_dim=ONT_DIM, 
    hidden_dim=768 
)

# Kiểm tra cấu trúc layer đầu tiên để chắc chắn nó là 24->768
print(f"🧐 Cấu trúc lớp ẩn: {model_wide_concat.ont_pipeline[0]}")

# Train
model_wide_concat = train_experiment(
    model_wide_concat, 
    "Wide Ont (768d) + Concat", 
    epochs=15, 
    patience=4
)
acc_c, f1_c = evaluate(model_wide_concat, test_loader)


# === CASE D: WIDE DEEP (768d) + GATE ===
print("\n" + "="*40)
print("🧪 CASE D: Wide Deep (24->768->768) + GATE")
print("="*40)

model_wide_gate = PhoBERT_DeepOntology(
    n_classes=NUM_CLASSES, 
    fusion_type='gate', 
    ont_input_dim=ONT_DIM, 
    hidden_dim=768
)

# Train
model_wide_gate = train_experiment(
    model_wide_gate, 
    "Wide Ont (768d) + Gate", 
    epochs=15, 
    patience=4
)
acc_d, f1_d = evaluate(model_wide_gate, test_loader)

# === TỔNG HỢP KẾT QUẢ MỚI ===
print(f"\n📊 KẾT QUẢ KIẾN TRÚC WIDE (24->768->768):")
print(f"1. Concat: F1-Macro = {f1_c:.4f} | Acc = {acc_c:.4f}")
print(f"2. Gate  : F1-Macro = {f1_d:.4f} | Acc = {acc_d:.4f}")

In [ ]:
import torch
import torch.nn as nn
from transformers import AutoModel

# ---------------------------------------------------------
# 1. ĐỊNH NGHĨA CLASS MODEL (Hỗ trợ tùy chỉnh hidden_dim)
# ---------------------------------------------------------

# Tự động lấy tham số môi trường
try:
    NUM_CLASSES = len(le.classes_)
    ONT_DIM = ONT_INPUT_DIM 
except NameError:
    NUM_CLASSES = 7
    ONT_DIM = 24 

from src.models.phobert_fusion_models import PhoBERT_DeepOntology

# ---------------------------------------------------------
# 2. CHẠY THỰC NGHIỆM: CASE BOTTLENECK (64d)
# ---------------------------------------------------------

# === CASE A: BOTTLENECK (64d) + CONCAT ===
print("\n" + "="*50)
print("🧪 CASE A: Bottleneck Deep (24->64->768) + CONCAT")
print("="*50)

# Khởi tạo model với hidden_dim = 64
model_bottleneck_concat = PhoBERT_DeepOntology(
    n_classes=NUM_CLASSES, 
    fusion_type='concat', 
    ont_input_dim=ONT_DIM, 
    hidden_dim=64  # <--- ĐIỂM QUAN TRỌNG: Nén xuống 64
)

# Kiểm tra in ra cấu trúc để chắc chắn
print(f"🧐 Cấu trúc lớp ẩn: {model_bottleneck_concat.ont_pipeline[0]}") 
# Mong đợi: Linear(in_features=24, out_features=64, bias=True)

model_bottleneck_concat = train_experiment(
    model_bottleneck_concat, 
    "Bottleneck (64d) + Concat", 
    epochs=15, 
    patience=4
)
acc_a, f1_a = evaluate(model_bottleneck_concat, test_loader)


# === CASE B: BOTTLENECK (64d) + GATE ===
print("\n" + "="*50)
print("🧪 CASE B: Bottleneck Deep (24->64->768) + GATE")
print("="*50)

model_bottleneck_gate = PhoBERT_DeepOntology(
    n_classes=NUM_CLASSES, 
    fusion_type='gate', 
    ont_input_dim=ONT_DIM, 
    hidden_dim=64  # <--- Nén xuống 64
)

model_bottleneck_gate = train_experiment(
    model_bottleneck_gate, 
    "Bottleneck (64d) + Gate", 
    epochs=15, 
    patience=4
)
acc_b, f1_b = evaluate(model_bottleneck_gate, test_loader)

# === TỔNG HỢP KẾT QUẢ ===
print(f"\n📊 KẾT QUẢ KIẾN TRÚC BOTTLENECK (24->64->768):")
print(f"1. Concat: F1-Macro = {f1_a:.4f} | Acc = {acc_a:.4f}")
print(f"2. Gate  : F1-Macro = {f1_b:.4f} | Acc = {acc_b:.4f}")

In [ ]:
# =========================================================
# CELL: EVALUATION - BASELINE vs. NEURO-SYMBOLIC (VSMEC 7-LABELS)
# Logic: Sử dụng m1 + Rule-based Post-processing từ Ontology
# =========================================================
import torch
import torch.nn.functional as F
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score, classification_report
from tqdm.auto import tqdm

class NeuroSymbolicReasonerVSMEC:
    def __init__(self, model, engine, tokenizer, device, 
                 ontology_threshold=0.5, model_conf_limit=0.7):
        self.model = model
        self.engine = engine
        self.tokenizer = tokenizer
        self.device = device
        self.ont_threshold = ontology_threshold
        self.model_conf_limit = model_conf_limit
        # Map index của model về nhãn cảm xúc thực tế
        self.idx2label = {i: label for i, label in enumerate(le.classes_)}

    def infer(self, text):
        # B1: Neural Inference (Dùng model m1)
        inputs = self.tokenizer(text, return_tensors="pt", truncation=True, max_length=128, padding='max_length')
        input_ids = inputs['input_ids'].to(self.device)
        attention_mask = inputs['attention_mask'].to(self.device)
        
        # Lấy vector KG từ Ontology Engine
        # Fix lỗi unpack: getvector trả về (fin, flat, hit_trace) khi debug=True
        vec, _, _ = self.engine.getvector(text, debug=True)
        ont_tensor = torch.tensor(np.array([vec]), dtype=torch.float).to(self.device)
        
        self.model.eval()
        with torch.no_grad():
            # M1 là PhoBERT không dùng feature ontology nên ta chỉ truyền ids và mask
            # Lưu ý: Theo cấu trúc PhoBERT_Fusion_V2, nếufusion_type='none' thì tham số thứ 3 (ont) không dùng
            outputs = self.model(input_ids, attention_mask, ont_tensor)
            probs = F.softmax(outputs, dim=1)
            
        pred_idx = torch.argmax(probs).item()
        confidence = probs[0][pred_idx].item()
        neural_label = self.idx2label[pred_idx]

        # B2: Symbolic Override (Hậu xử lý bằng luật)
        # Lấy cảm xúc mạnh nhất từ Ontology (7 chiều đầu của vector)
        ont_emotions = vec[0:7]
        max_ont_idx = np.argmax(ont_emotions)
        max_ont_score = ont_emotions[max_ont_idx]
        ont_label_name = self.engine.ALLEMOTIONS[max_ont_idx]
        
        # Mapping nhãn từ Ontology sang nhãn VSMEC (Happiness -> Enjoyment, Neutral -> Other)
        ont_to_vsmec = {
            "Happiness": "Enjoyment",
            "Neutral": "Other"
        }
        final_ont_label = ont_to_vsmec.get(ont_label_name, ont_label_name)
        
        final_label = neural_label
        
        # RULE: Nếu Model dự đoán là 'Other' nhưng Ontology bắt được cảm xúc cụ thể rất mạnh
        # Hoặc Model thiếu tự tin (confidence thấp) và Ontology có tín hiệu rõ ràng
        if neural_label == "Other" or confidence < self.model_conf_limit:
            if max_ont_score > self.ont_threshold:
                # Tìm index của nhãn Ontology trong danh sách nhãn VSMEC
                try:
                    target_idx = list(le.classes_).index(final_ont_label)
                    final_label = final_ont_label
                except ValueError:
                    pass # Nhãn không tồn tại trong VSMEC thì bỏ qua

        return neural_label, final_label

# 2. Chạy Đánh giá trên tập Test của VSMEC
print(f"🚀 Bắt đầu đánh giá Neuro-Symbolic trên {len(df_test)} mẫu test VSMEC...")

# Khởi tạo Reasoner với model Baseline (m1)
reasoner_vsmec = NeuroSymbolicReasonerVSMEC(
    model=m1, 
    engine=ontengine, 
    tokenizer=tokenizer, 
    device=device,
    ontology_threshold=0.4, # Ngưỡng nhạy hơn cho 7 nhãn
    model_conf_limit=0.65
)

y_true = []
y_pred_base = []
y_pred_neuro = []

for _, row in tqdm(df_test.iterrows(), total=len(df_test)):
    text = row['Sentence']
    true_label_str = row['Emotion']
    
    base_lbl, neuro_lbl = reasoner_vsmec.infer(text)
    
    y_true.append(true_label_str)
    y_pred_base.append(base_lbl)
    y_pred_neuro.append(neuro_lbl)

# 3. Tính Metrics & Báo cáo
acc_base = accuracy_score(y_true, y_pred_base)
f1_base = f1_score(y_true, y_pred_base, average='macro')

acc_neuro = accuracy_score(y_true, y_pred_neuro)
f1_neuro = f1_score(y_true, y_pred_neuro, average='macro')

results_df = pd.DataFrame({
    "Metric": ["Accuracy", "F1-Macro"],
    "Baseline (m1)": [acc_base, f1_base],
    "M1 + Ontology Rules": [acc_neuro, f1_neuro],
    "Improvement": [acc_neuro - acc_base, f1_neuro - f1_base]
})

print("\n" + "="*60)
print("🏆 KẾT QUẢ VSMEC: BASELINE vs RULE-BASED POST-PROCESSING")
print("="*60)
print(results_df.round(4).to_markdown(index=False))

print("\n🧠 Chi tiết M1 + Ontology Rules:")
print(classification_report(y_true, y_pred_neuro, target_names=le.classes_, digits=4))

# Đếm số ca đã sửa
changes = sum(1 for b, n in zip(y_pred_base, y_pred_neuro) if b != n)
print(f"\n🔄 Tổng số mẫu đã được Ontology can thiệp sửa lại: {changes}/{len(df_test)} ({changes/len(df_test)*100:.2f}%)")